# 02 FHIR Patient Silver — Silver Layer

Read the `patient` Bronze Delta table, parse the original FHIR JSON, clean and deduplicate records, save the Silver table, and maintain SCD Type 2 history.

**Prerequisite:** Run the matching `Patient` ingestion notebook first and attach `FHIR_Lakehouse`.

**Bronze design:** The Bronze table stores the complete FHIR resource in `resource_json` plus extraction/load metadata. This notebook parses the JSON before accessing resource fields.

In [1]:
from pyspark.sql.functions import (
    col, lit, trim, lower, when, row_number,
    sha2, coalesce, from_json, get_json_object,
    current_timestamp, to_date
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# READ BRONZE DATA
bronze_patient_df = spark.read.table("bronze.patient")

print("Bronze Patient loaded!")
print("Bronze records:", bronze_patient_df.count())
print("Bronze columns:", bronze_patient_df.columns)

StatementMeta(, 28c76fd6-0522-42eb-8f95-1e79bfc8325f, 3, Finished, Available, Finished, False)

Bronze Patient loaded!
Bronze records: 1485
Bronze columns: ['id', 'resourceType', 'resource_json', 'extraction_timestamp', 'api_url_or_params', 'load_timestamp', 'source_file']


In [2]:
# PARSE THE ORIGINAL FHIR JSON

# Infer the resource schema from the JSON records in Bronze.
resource_schema = spark.read.json(
    bronze_patient_df.select("resource_json").rdd.map(lambda row: row["resource_json"])
).schema

silver_patient_df = (
    bronze_patient_df
    .withColumn(
        "resource",
        from_json(col("resource_json"), resource_schema)
    )
    .select(
        "resource.*",
        "resource_json",
        "extraction_timestamp",
        "api_url_or_params",
        "load_timestamp",
        "source_file"
    )
)

print("FHIR JSON parsed into resource columns.")
silver_patient_df.printSchema()

StatementMeta(, 28c76fd6-0522-42eb-8f95-1e79bfc8325f, 4, Finished, Available, Finished, False)

FHIR JSON parsed into resource columns.
root
 |-- _active: struct (nullable = true)
 |    |-- extension: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- url: string (nullable = true)
 |    |    |    |-- valueString: string (nullable = true)
 |    |-- id: string (nullable = true)
 |-- _birthDate: struct (nullable = true)
 |    |-- extension: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- url: string (nullable = true)
 |    |    |    |-- valueDateTime: string (nullable = true)
 |-- active: boolean (nullable = true)
 |-- address: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- city: string (nullable = true)
 |    |    |-- country: string (nullable = true)
 |    |    |-- district: string (nullable = true)
 |    |    |-- extension: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- extension: array (

In [3]:
# CLEAN PATIENT DATA

silver_patient_df = silver_patient_df.filter(
    col("id").isNotNull() & (trim(col("id")) != "")
).filter(col("resourceType") == "Patient")

silver_patient_df = silver_patient_df.withColumn(
    "gender", lower(trim(col("gender")))
).withColumn(
    "gender",
    when(
        col("gender").isin("male", "female", "other", "unknown"),
        col("gender")
    ).otherwise("unknown")
).withColumn(
    "birthDate",
    when(
        to_date(col("birthDate"), "yyyy-MM-dd").isNotNull(),
        col("birthDate")
    ).otherwise(lit(None).cast("string"))
)

print("Patient cleaning completed!")

StatementMeta(, 28c76fd6-0522-42eb-8f95-1e79bfc8325f, 5, Finished, Available, Finished, False)

Patient cleaning completed!


In [4]:
# DEDUPLICATE BY RESOURCE ID

window_spec = (
    Window
    .partitionBy("id")
    .orderBy(
        col("extraction_timestamp").desc(),
        col("load_timestamp").desc()
    )
)

silver_patient_df = (
    silver_patient_df
    .withColumn("row_number", row_number().over(window_spec))
    .filter(col("row_number") == 1)
    .drop("row_number")
)

print("Bronze records:", bronze_patient_df.count())
print("Silver unique records:", silver_patient_df.count())

StatementMeta(, 28c76fd6-0522-42eb-8f95-1e79bfc8325f, 6, Finished, Available, Finished, False)

Bronze records: 1485
Silver unique records: 1484


In [5]:
# SAVE SILVER TABLE

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

(
    silver_patient_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.patient")
)

print("Saved silver.patient")
print("Silver records:", spark.table("silver.patient").count())

StatementMeta(, 28c76fd6-0522-42eb-8f95-1e79bfc8325f, 7, Finished, Available, Finished, False)

Saved silver.patient
Silver records: 1484


In [6]:
# INITIALIZE SCD TYPE 2 HISTORY IF IT DOES NOT EXIST

history_table = "silver.patient_history"

if not spark.catalog.tableExists(history_table):
    initial_history_df = (
        spark.table("silver.patient")
        .withColumn(
            "record_hash",
            sha2(coalesce(col("resource_json"), lit("")), 256)
        )
        .withColumn(
            "effective_start_date",
            col("extraction_timestamp").cast("timestamp")
        )
        .withColumn(
            "effective_end_date",
            lit(None).cast("timestamp")
        )
        .withColumn("is_current", lit(True))
    )

    (
        initial_history_df.write
        .format("delta")
        .mode("errorifexists")
        .saveAsTable(history_table)
    )

    print("Initial SCD Type 2 history created:", history_table)
else:
    print("History table already exists; existing history will be preserved.")

history_df = spark.table(history_table)
print("History records:", history_df.count())
history_df.groupBy("is_current").count().show()

StatementMeta(, 28c76fd6-0522-42eb-8f95-1e79bfc8325f, 8, Finished, Available, Finished, False)

History table already exists; existing history will be preserved.
History records: 1846
+----------+-----+
|is_current|count|
+----------+-----+
|      true| 1831|
|     false|   15|
+----------+-----+



In [7]:
# DETECT NEW OR CHANGED RECORDS

incoming_df = (
    spark.table("silver.patient")
    .withColumn(
        "record_hash",
        sha2(coalesce(col("resource_json"), lit("")), 256)
    )
)

current_history_df = (
    spark.table(history_table)
    .filter(col("is_current") == True)
    .select("id", "record_hash")
)

comparison_df = (
    incoming_df.alias("src")
    .join(current_history_df.alias("tgt"), on="id", how="left")
)

changed_or_new_df = comparison_df.filter(
    col("tgt.record_hash").isNull() |
    (col("src.record_hash") != col("tgt.record_hash"))
)

new_versions_df = changed_or_new_df.select("src.*")

print("Incoming records:", incoming_df.count())
print("Existing current records:", current_history_df.count())
print("New or changed records:", new_versions_df.count())

StatementMeta(, 28c76fd6-0522-42eb-8f95-1e79bfc8325f, 9, Finished, Available, Finished, False)

Incoming records: 1484
Existing current records: 1831
New or changed records: 33


In [8]:
# APPLY SCD TYPE 2 CHANGES

changed_ids_df = new_versions_df.select("id").distinct()
changed_count = new_versions_df.count()

if changed_count > 0:
    run_timestamp = spark.sql(
        "SELECT current_timestamp() AS ts"
    ).first()["ts"]

    delta_history = DeltaTable.forName(spark, history_table)

    (
        delta_history.alias("tgt")
        .merge(
            changed_ids_df.alias("src"),
            "tgt.id = src.id AND tgt.is_current = true"
        )
        .whenMatchedUpdate(set={
            "is_current": lit(False),
            "effective_end_date": lit(run_timestamp).cast("timestamp")
        })
        .execute()
    )

    new_history_df = (
        new_versions_df
        .withColumn(
            "effective_start_date",
            lit(run_timestamp).cast("timestamp")
        )
        .withColumn(
            "effective_end_date",
            lit(None).cast("timestamp")
        )
        .withColumn("is_current", lit(True))
    )

    (
        new_history_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(history_table)
    )

    print("SCD Type 2 changes applied:", changed_count)
else:
    print("No new or changed records. History remains unchanged.")

StatementMeta(, 28c76fd6-0522-42eb-8f95-1e79bfc8325f, 10, Finished, Available, Finished, False)

SCD Type 2 changes applied: 33


In [9]:
# VALIDATE HISTORY TABLE

history_df = spark.table(history_table)

print("Total history records:", history_df.count())
history_df.groupBy("is_current").count().show()

duplicate_current_ids = (
    history_df
    .filter(col("is_current") == True)
    .groupBy("id")
    .count()
    .filter(col("count") > 1)
)

print("IDs with more than one current version:", duplicate_current_ids.count())
duplicate_current_ids.show(20, truncate=False)

StatementMeta(, 28c76fd6-0522-42eb-8f95-1e79bfc8325f, 11, Finished, Available, Finished, False)

Total history records: 1879
+----------+-----+
|is_current|count|
+----------+-----+
|      true| 1863|
|     false|   16|
+----------+-----+

IDs with more than one current version: 0
+---+-----+
|id |count|
+---+-----+
+---+-----+



In [10]:
# FINAL SILVER VALIDATION

spark.table("silver.patient").select(
    "id", "resourceType", "extraction_timestamp", "load_timestamp"
).show(10, truncate=False)

StatementMeta(, 28c76fd6-0522-42eb-8f95-1e79bfc8325f, 12, Finished, Available, Finished, False)

+------------------------------------+------------+--------------------------+--------------------------+
|id                                  |resourceType|extraction_timestamp      |load_timestamp            |
+------------------------------------+------------+--------------------------+--------------------------+
|0193a000-0000-7000-8000-000000000001|Patient     |2026-11-18 17:41:12.73253 |2026-11-18 17:46:09.709527|
|1170                                |Patient     |2026-11-18 17:41:12.73253 |2026-11-18 17:46:09.709527|
|166914                              |Patient     |2026-11-18 17:41:12.73253 |2026-11-18 17:46:09.709527|
|38016                               |Patient     |2026-11-18 19:37:38.364437|2026-11-18 19:38:10.434091|
|399ce81c-df96-4f3e-bacc-61f4f1c2c982|Patient     |2026-11-18 17:41:12.73253 |2026-11-18 17:46:09.709527|
|492147                              |Patient     |2026-11-18 17:41:12.73253 |2026-11-18 17:46:09.709527|
|492208                              |Patient 